# 01. 어린이교통사고 수집 및 전처리

도로교통공단 API에서 2020~2025년 교통사고 통계를 수집하고,
`어린이사고`만 추출해 229개 지역 × 6개 연도의 완전한 패널 형태로 정리합니다.

**출력**
- `../data_raw/accident_2020_2025_raw.csv`
- `../data_clean/accident_2020_2025_child_clean.csv`

In [1]:
import requests
import pandas as pd
import time
from urllib.parse import unquote

# 공공데이터포털 Encoding 인증키를 입력하세요.
ENCODING_KEY = "dJ86KOJSsPyb6606Leb2zS7oit1l%2BHE1Vo2nmwAzyHvB7I1UdP%2B0guI2VeyfG9zFcQS7pvg2Ml11qoh8LcRIoA%3D%3D"
service_key = unquote(ENCODING_KEY)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
BASE_URL = "https://apis.data.go.kr/B552061/lgStat/getRestLgStat"

SIDO_CODES = [
    "1100", "1200", "1300", "1400", "1500", "1600",
    "1700", "1800", "1900", "2000", "2100", "2200",
    "2300", "2400", "2500", "2600", "2700"
]


## 1) 2020~2025 전체 사고 통계 수집

In [3]:
all_rows = []

for year in YEARS:
    print(f"\n===== {year}년 수집 시작 =====")
    region_count = 0

    for sido in SIDO_CODES:
        prefix = sido[:2]

        for num in range(1, 41):
            gugun = f"{prefix}{num:02d}"

            params = {
                "serviceKey": service_key,
                "searchYearCd": year,
                "siDo": sido,
                "guGun": gugun,
                "type": "json",
                "numOfRows": 100,
                "pageNo": 1,
            }

            try:
                response = requests.get(BASE_URL, params=params, timeout=15)
                response.raise_for_status()
                data = response.json()

                if data.get("resultCode") == "00":
                    items = data.get("items", {}).get("item", [])
                    if isinstance(items, dict):
                        items = [items]

                    if items:
                        all_rows.extend(items)
                        region_count += 1

            except Exception as e:
                print("오류:", year, sido, gugun, e)

            time.sleep(0.05)

        print(f"{year}년 / 시도코드 {sido} 완료")

    print(f"{year}년 수집 지역 수: {region_count}")

print("\n전체 수집 끝!")


===== 2020년 수집 시작 =====
2020년 / 시도코드 1100 완료
2020년 / 시도코드 1200 완료
2020년 / 시도코드 1300 완료
2020년 / 시도코드 1400 완료
2020년 / 시도코드 1500 완료
2020년 / 시도코드 1600 완료
2020년 / 시도코드 1700 완료
2020년 / 시도코드 1800 완료
2020년 / 시도코드 1900 완료
2020년 / 시도코드 2000 완료
2020년 / 시도코드 2100 완료
2020년 / 시도코드 2200 완료
2020년 / 시도코드 2300 완료
2020년 / 시도코드 2400 완료
2020년 / 시도코드 2500 완료
2020년 / 시도코드 2600 완료
2020년 / 시도코드 2700 완료
2020년 수집 지역 수: 229

===== 2021년 수집 시작 =====
2021년 / 시도코드 1100 완료
2021년 / 시도코드 1200 완료
2021년 / 시도코드 1300 완료
2021년 / 시도코드 1400 완료
2021년 / 시도코드 1500 완료
2021년 / 시도코드 1600 완료
2021년 / 시도코드 1700 완료
2021년 / 시도코드 1800 완료
2021년 / 시도코드 1900 완료
2021년 / 시도코드 2000 완료
2021년 / 시도코드 2100 완료
2021년 / 시도코드 2200 완료
2021년 / 시도코드 2300 완료
2021년 / 시도코드 2400 완료
2021년 / 시도코드 2500 완료
2021년 / 시도코드 2600 완료
2021년 / 시도코드 2700 완료
2021년 수집 지역 수: 229

===== 2022년 수집 시작 =====
2022년 / 시도코드 1100 완료
2022년 / 시도코드 1200 완료
2022년 / 시도코드 1300 완료
2022년 / 시도코드 1400 완료
2022년 / 시도코드 1500 완료
2022년 / 시도코드 1600 완료
2022년 / 시도코드 1700 완료
2022년 / 시도코드 1800 완료
2022년

In [4]:
accident_raw = pd.DataFrame(all_rows)

accident_raw["std_year"] = pd.to_numeric(
    accident_raw["std_year"], errors="coerce"
).astype("Int64")

print("전체 크기:", accident_raw.shape)
print("\n연도별 행 개수")
print(accident_raw["std_year"].value_counts().sort_index())

print("\n연도별 지역 수")
print(accident_raw.groupby("std_year")["sido_sgg_nm"].nunique())

display(accident_raw.head())

전체 크기: (16621, 27)

연도별 행 개수
std_year
2020    2752
2021    2770
2022    2797
2023    2772
2024    2757
2025    2773
Name: count, dtype: Int64

연도별 지역 수
std_year
2020    229
2021    229
2022    229
2023    230
2024    229
2025    229
Name: sido_sgg_nm, dtype: int64


,std_year,sido_sgg_nm,acc_cl_nm,acc_cnt,acc_cnt_cmrt,dth_dnv_cnt,dth_dnv_cnt_cmrt,ftlt_rate,injpsn_cnt,injpsn_cnt_cmrt,...,cnt_027_03,cnt_027_04,cnt_027_05,cnt_027_06,cnt_027_07,cnt_027_99,cnt_014_01,cnt_014_02,cnt_014_03,cnt_014_04
0,2020,서울특별시 종로구,전체사고,903,0.43,5,0.16,0.55,1224,0.40,...,108,166,463,30,35,70,223,648,32,0
1,2020,서울특별시 종로구,어린이사고,16,0.19,0,0.00,0.00,19,0.18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020,서울특별시 종로구,고령자사고,147,0.42,2,0.15,1.36,153,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020,서울특별시 종로구,보행자사고,226,0.62,2,0.18,0.88,234,0.63,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020,서울특별시 종로구,자전거사고,20,0.35,1,1.20,5.00,23,0.37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
accident_raw.to_csv(
    "../data_raw/accident_2020_2025_raw.csv",
    index=False,
    encoding="utf-8-sig"
)
print("원본 저장 완료")

원본 저장 완료


## 2) 어린이사고 추출 및 사고 0건 지역 보완

In [6]:
child = accident_raw[
    accident_raw["acc_cl_nm"] == "어린이사고"
].copy()

base_regions = (
    accident_raw[["std_year", "sido_sgg_nm"]]
    .drop_duplicates()
)

child_values = child[
    ["std_year", "sido_sgg_nm", "acc_cnt", "dth_dnv_cnt", "injpsn_cnt"]
].copy()

accident_child = base_regions.merge(
    child_values,
    on=["std_year", "sido_sgg_nm"],
    how="left"
)

num_cols = ["acc_cnt", "dth_dnv_cnt", "injpsn_cnt"]
for col in num_cols:
    accident_child[col] = pd.to_numeric(
        accident_child[col], errors="coerce"
    ).fillna(0)

print("보완 후 크기:", accident_child.shape)
print(accident_child.groupby("std_year")["sido_sgg_nm"].nunique())

보완 후 크기: (1375, 5)
std_year
2020    229
2021    229
2022    229
2023    230
2024    229
2025    229
Name: sido_sgg_nm, dtype: int64


## 3) 행정구역 변경 보정 및 최종 저장

In [7]:
# 군위군의 소속 변경을 현재 분석 기준에 맞춤
accident_child["sido_sgg_nm"] = accident_child["sido_sgg_nm"].replace({
    "경상북도 군위군(구)": "대구광역시 군위군"
})

# 같은 연도·지역이 된 행은 합산
accident_child = (
    accident_child
    .groupby(["std_year", "sido_sgg_nm"], as_index=False)[
        ["acc_cnt", "dth_dnv_cnt", "injpsn_cnt"]
    ]
    .sum()
)

accident_child[["sido_nm", "sgg_nm"]] = (
    accident_child["sido_sgg_nm"]
    .str.split(" ", n=1, expand=True)
)

# 세종 예외 처리
accident_child.loc[
    accident_child["sido_nm"] == "세종특별자치시",
    "sgg_nm"
] = "세종특별자치시"

print("결측치")
print(accident_child.isna().sum())
print("\n중복 행:", accident_child.duplicated().sum())
print("\n연도별 지역 수")
print(accident_child.groupby("std_year")["sido_sgg_nm"].nunique())

결측치
std_year       0
sido_sgg_nm    0
acc_cnt        0
dth_dnv_cnt    0
injpsn_cnt     0
sido_nm        0
sgg_nm         0
dtype: int64

중복 행: 0

연도별 지역 수
std_year
2020    229
2021    229
2022    229
2023    229
2024    229
2025    229
Name: sido_sgg_nm, dtype: int64


In [8]:
accident_child = accident_child.reset_index(drop=True)

accident_child.to_csv(
    "../data_clean/accident_2020_2025_child_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", accident_child.shape)
display(accident_child.head())

저장 완료: (1374, 7)


,std_year,sido_sgg_nm,acc_cnt,dth_dnv_cnt,injpsn_cnt,sido_nm,sgg_nm
0,2020,강원도 강릉시,40.0,0.0,49.0,강원도,강릉시
1,2020,강원도 고성군,9.0,0.0,10.0,강원도,고성군
2,2020,강원도 동해시,19.0,0.0,25.0,강원도,동해시
3,2020,강원도 삼척시,15.0,0.0,16.0,강원도,삼척시
4,2020,강원도 속초시,25.0,0.0,34.0,강원도,속초시
